In [12]:
# Display summary
print("\n" + "=" * 80)
print("📊 SUMMARY")
print("=" * 80)

print(f"\nSampling Statistics:")
print(f"  Total articles in dataset: {len(merged):,}")
print(f"  Sampled for test: {len(test_set)} ({len(test_set)/len(merged)*100:.2f}%)")
print(f"  Sampled for validation: {len(validation_set)} ({len(validation_set)/len(merged)*100:.2f}%)")
print(f"  Total sampled: {len(combined_sample)} ({len(combined_sample)/len(merged)*100:.2f}%)")

print(f"\nCategory Representation Check:")
comparison = pd.DataFrame({
    'Dataset': merged['final_category'].value_counts(normalize=True) * 100,
    'Test': test_set['final_category'].value_counts(normalize=True) * 100,
    'Validation': validation_set['final_category'].value_counts(normalize=True) * 100
}).round(1)
print(comparison.to_string())

print(f"\n✅ Sampling complete! Ready for manual labeling.")
print(f"\n📝 Next steps:")
print(f"   1. Label the articles in: {COMBINED_FOR_REVIEW}")
print(f"   2. After labeling, use notebook 22_cross_validation_v2 to evaluate performance")
print(f"   3. Compare rule-based predictions vs actual labels")


📊 SUMMARY

Sampling Statistics:
  Total articles in dataset: 19,659
  Sampled for test: 400 (2.03%)
  Sampled for validation: 300 (1.53%)
  Total sampled: 700 (3.56%)

Category Representation Check:
                Dataset  Test  Validation
final_category                           
Governance         62.7  62.7        62.7
Personnel           8.4   8.5         8.3
Products            7.3   7.2         7.3
IT/Data             7.1   7.2         7.0
No Event            6.6   6.5         6.7
Communication       4.1   4.0         4.3
Legal               2.4   2.5         2.3
Processes           1.3   1.2         1.3

✅ Sampling complete! Ready for manual labeling.

📝 Next steps:
   1. Label the articles in: data/samples/combined_test_val_for_labeling_v4.csv
   2. After labeling, use notebook 22_cross_validation_v2 to evaluate performance
   3. Compare rule-based predictions vs actual labels


## Summary Statistics

In [11]:
import os

# Create output directory if needed
os.makedirs('data/samples', exist_ok=True)

# Save individual sets (without labels - reference only)
test_set.to_csv(TEST_SET_OUTPUT.replace('.csv', '_unlabeled.csv'), index=False)
validation_set.to_csv(VALIDATION_SET_OUTPUT.replace('.csv', '_unlabeled.csv'), index=False)

# Save combined file for labeling (main file to use)
combined_for_review.to_csv(COMBINED_FOR_REVIEW, index=False)

# Save separate review files (alternative if you want to label separately)
test_for_review.to_csv(TEST_SET_OUTPUT.replace('.csv', '_for_labeling.csv'), index=False)
val_for_review.to_csv(VALIDATION_SET_OUTPUT.replace('.csv', '_for_labeling.csv'), index=False)

print("\n" + "=" * 80)
print("✅ FILES SAVED")
print("=" * 80)

print(f"\n📁 Main file for labeling (recommended):")
print(f"   {COMBINED_FOR_REVIEW}")
print(f"   → {len(combined_for_review)} articles ({len(test_for_review)} test + {len(val_for_review)} validation)")

print(f"\n📁 Separate files (if you prefer to label separately):")
print(f"   Test: {TEST_SET_OUTPUT.replace('.csv', '_for_labeling.csv')}")
print(f"   Validation: {VALIDATION_SET_OUTPUT.replace('.csv', '_for_labeling.csv')}")

print(f"\n📋 Instructions for manual labeling:")
print(f"   1. Open: {COMBINED_FOR_REVIEW}")
print(f"   2. For each article, fill in the 'actual_label' column with one of:")
print(f"      - Governance")
print(f"      - Personnel") 
print(f"      - Products")
print(f"      - IT/Data")
print(f"      - Processes")
print(f"      - Legal")
print(f"      - Communication")
print(f"      - No Event")
print(f"   3. Optionally add notes in the 'notes' column")
print(f"   4. The 'provisional_category' is the rule-based prediction (for reference)")
print(f"   5. Save the file with '_labeled' suffix when complete")

print(f"\n💡 After labeling, you can split the file back into test/validation using the 'set_type' column")


✅ FILES SAVED

📁 Main file for labeling (recommended):
   data/samples/combined_test_val_for_labeling_v4.csv
   → 700 articles (400 test + 300 validation)

📁 Separate files (if you prefer to label separately):
   Test: data/samples/test_set_v4_for_labeling.csv
   Validation: data/samples/validation_set_v4_for_labeling.csv

📋 Instructions for manual labeling:
   1. Open: data/samples/combined_test_val_for_labeling_v4.csv
   2. For each article, fill in the 'actual_label' column with one of:
      - Governance
      - Personnel
      - Products
      - IT/Data
      - Processes
      - Legal
      - Communication
      - No Event
   3. Optionally add notes in the 'notes' column
   4. The 'provisional_category' is the rule-based prediction (for reference)
   5. Save the file with '_labeled' suffix when complete

💡 After labeling, you can split the file back into test/validation using the 'set_type' column


## Save Files

In [10]:
def prepare_for_labeling(df, set_name):
    """
    Prepare dataset for manual labeling.
    
    Args:
        df: DataFrame with sampled articles
        set_name: Name of the set (for tracking)
    
    Returns:
        DataFrame formatted for manual review
    """
    review_df = df[[
        'article_id',
        'title', 
        'summary',
        'final_category',
        'provisional_label',
        'most_relevant_keywords',
        'published_date',
        'url'
    ]].copy()
    
    # Rename for clarity
    review_df = review_df.rename(columns={
        'final_category': 'provisional_category',
        'most_relevant_keywords': 'top_keywords'
    })
    
    # Add empty column for manual labeling
    review_df['actual_label'] = ''
    
    # Add notes column for reviewer comments
    review_df['notes'] = ''
    
    # Add set identifier
    review_df['set_type'] = set_name
    
    # Reorder columns for easier review
    review_df = review_df[[
        'set_type',
        'article_id',
        'title',
        'summary',
        'provisional_category',
        'provisional_label',
        'top_keywords',
        'actual_label',
        'notes',
        'published_date',
        'url'
    ]]
    
    return review_df

# Prepare both sets
test_for_review = prepare_for_labeling(test_set, 'TEST')
val_for_review = prepare_for_labeling(validation_set, 'VALIDATION')

# Combine for single labeling session
combined_for_review = pd.concat([test_for_review, val_for_review], ignore_index=True)

print(f"\n✓ Prepared {len(test_for_review)} test articles for review")
print(f"✓ Prepared {len(val_for_review)} validation articles for review")
print(f"✓ Combined: {len(combined_for_review)} articles total")


✓ Prepared 400 test articles for review
✓ Prepared 300 validation articles for review
✓ Combined: 700 articles total


## Prepare for Manual Labeling

Create CSV files with columns needed for manual review:
- article_id
- title
- summary
- provisional_category (from rule-based classification)
- url (for reference)
- actual_label (empty - to be filled in during review)

In [9]:
# Split into test and validation maintaining stratification
print("\n" + "=" * 80)
print("STEP 2: Split into test and validation sets")
print("=" * 80)

test_set, validation_set = train_test_split(
    combined_sample,
    test_size=VALIDATION_SET_SIZE / TOTAL_SAMPLE_SIZE,
    stratify=combined_sample['final_category'],
    random_state=RANDOM_SEED
)

print(f"\n✓ Test set: {len(test_set)} articles")
print(f"✓ Validation set: {len(validation_set)} articles")

print(f"\n📊 Test Set Distribution:")
print(test_set['final_category'].value_counts().to_string())

print(f"\n📊 Validation Set Distribution:")
print(validation_set['final_category'].value_counts().to_string())


STEP 2: Split into test and validation sets

✓ Test set: 400 articles
✓ Validation set: 300 articles

📊 Test Set Distribution:
final_category
Governance       251
Personnel         34
Products          29
IT/Data           29
No Event          26
Communication     16
Legal             10
Processes          5

📊 Validation Set Distribution:
final_category
Governance       188
Personnel         25
Products          22
IT/Data           21
No Event          20
Communication     13
Legal              7
Processes          4


## Split into Test and Validation Sets

In [8]:
def stratified_sample(df, n_samples, stratify_col, random_state=42):
    """
    Perform stratified sampling to ensure all categories are represented.
    
    Args:
        df: DataFrame to sample from
        n_samples: Total number of samples to draw
        stratify_col: Column to stratify on
        random_state: Random seed
    
    Returns:
        Sampled DataFrame
    """
    # Calculate samples per category (proportional)
    category_counts = df[stratify_col].value_counts()
    category_proportions = category_counts / len(df)
    
    samples_per_category = (category_proportions * n_samples).round().astype(int)
    
    # Adjust to ensure exact total (handle rounding)
    diff = n_samples - samples_per_category.sum()
    if diff != 0:
        # Add/subtract from largest category
        largest_cat = samples_per_category.idxmax()
        samples_per_category[largest_cat] += diff
    
    print(f"\n🎯 Sampling {n_samples} articles (stratified by {stratify_col}):")
    for cat, count in samples_per_category.items():
        pct = (count / n_samples) * 100
        available = len(df[df[stratify_col] == cat])
        print(f"   {cat:20s}: {count:4d} samples ({pct:5.1f}%) from {available:5d} available")
    
    # Sample from each category
    sampled_dfs = []
    for category, n in samples_per_category.items():
        category_df = df[df[stratify_col] == category]
        
        if len(category_df) < n:
            print(f"⚠️  Warning: {category} has only {len(category_df)} articles, sampling all")
            sampled_dfs.append(category_df)
        else:
            sampled = category_df.sample(n=n, random_state=random_state)
            sampled_dfs.append(sampled)
    
    return pd.concat(sampled_dfs, ignore_index=True).sample(frac=1, random_state=random_state)

# First, sample total amount from full dataset
print("\n" + "=" * 80)
print("STEP 1: Sample combined test + validation set")
print("=" * 80)

combined_sample = stratified_sample(
    merged, 
    TOTAL_SAMPLE_SIZE, 
    'final_category', 
    RANDOM_SEED
)

print(f"\n✓ Sampled {len(combined_sample)} articles total")


STEP 1: Sample combined test + validation set

🎯 Sampling 700 articles (stratified by final_category):
   Governance          :  439 samples ( 62.7%) from 12326 available
   Personnel           :   59 samples (  8.4%) from  1660 available
   Products            :   51 samples (  7.3%) from  1434 available
   IT/Data             :   50 samples (  7.1%) from  1405 available
   No Event            :   46 samples (  6.6%) from  1304 available
   Communication       :   29 samples (  4.1%) from   810 available
   Legal               :   17 samples (  2.4%) from   465 available
   Processes           :    9 samples (  1.3%) from   255 available

✓ Sampled 700 articles total


## Stratified Sampling

Sample articles proportionally from each provisional category to ensure representation of all categories in test/validation sets.

In [3]:
# Load provisionally labeled data
print("\n📂 Loading data...")
provisionally_labeled = pd.read_parquet(PROVISIONALLY_LABELED_DATA)
articles = pd.read_parquet(ARTICLES_DATA)

print(f"✓ Loaded {len(provisionally_labeled)} provisionally labeled articles")
print(f"✓ Loaded {len(articles)} article metadata")

# Merge to get full article information
merged = provisionally_labeled.merge(
    articles[['article_id', 'title', 'summary', 'published_date', 'url']], 
    on='article_id', 
    how='inner'
)

print(f"✓ Merged dataset: {len(merged)} articles with complete information")

# Show provisional category distribution
print(f"\n📊 Provisional Category Distribution:")
print(merged['final_category'].value_counts().to_string())
print(f"\nProvisional Label Types:")
print(merged['provisional_label'].value_counts().to_string())


📂 Loading data...
✓ Loaded 19659 provisionally labeled articles
✓ Loaded 19659 article metadata
✓ Merged dataset: 19659 articles with complete information

📊 Provisional Category Distribution:
final_category
Governance       12326
Personnel         1660
Products          1434
IT/Data           1405
No Event          1304
Communication      810
Legal              465
Processes          255

Provisional Label Types:
provisional_label
Complex Event (High Confidence, Low Delta)     16370
Low Confidence                                  1304
Complex Event (High Confidence, High Delta)     1008
Single Label                                     977


## Load Data

In [2]:
# Input data
PROVISIONALLY_LABELED_DATA = 'results/phase ii/21_provisionally_labeled_train_set_v4.parquet'
ARTICLES_DATA = 'data/01_preprocessed/articles_entity_linked_cleaned.parquet'

# Sample sizes
TEST_SET_SIZE = 400
VALIDATION_SET_SIZE = 300
TOTAL_SAMPLE_SIZE = TEST_SET_SIZE + VALIDATION_SET_SIZE

# Output paths
TEST_SET_OUTPUT = 'data/samples/test_set_v4.csv'
VALIDATION_SET_OUTPUT = 'data/samples/validation_set_v4.csv'
COMBINED_FOR_REVIEW = 'data/samples/combined_test_val_for_labeling_v4.csv'

# Random seed for reproducibility
RANDOM_SEED = 42

print(f"📊 Configuration:")
print(f"   Test set size: {TEST_SET_SIZE}")
print(f"   Validation set size: {VALIDATION_SET_SIZE}")
print(f"   Total to sample: {TOTAL_SAMPLE_SIZE}")
print(f"   Random seed: {RANDOM_SEED}")

📊 Configuration:
   Test set size: 400
   Validation set size: 300
   Total to sample: 700
   Random seed: 42


## Configuration

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

print("📚 Test & Validation Set Generator")
print("=" * 80)

📚 Test & Validation Set Generator


## Setup

# Create Test & Validation Sets for Manual Labeling

**Purpose**: Generate stratified samples from the cleaned entity-linked dataset for manual labeling.

**Process**:
1. Load provisionally labeled articles from notebook 21
2. Create stratified samples based on provisional categories
3. Export articles with metadata for manual review
4. Generate separate test and validation sets

**Target sizes**:
- Test set: 400 articles
- Validation set: 300 articles
- Total: 700 articles to label

**Stratification**: Sample proportionally from each provisional category + "No Event" articles